In [ ]:
import random
import numpy as np
import time
import numba
import math
from numpy.random import Generator, PCG64DXSM, SeedSequence
import concurrent.futures
from structure_ORIGINAL_FNs_singlesideBASEreinLEARNING2_2026_mod import play_sequence

np.set_printoptions(suppress=True)

In [ ]:
def make_linear_pairs(n_stim, hold_out=None):
    adj = []
    for i in range(n_stim - 1):
        adj.append([i, i+1])
        adj.append([i+1, i])
    adj = np.asarray(adj, dtype=np.int64)
    all_possible = []
    for i in range(n_stim):
        for j in range(n_stim):
            if i != j: all_possible.append([i, j])
    adj_set = set(tuple(p) for p in adj)
    non_adj = []
    test = []
    if hold_out is None:
        hold_out = [(1, 3), (3, 1)]
    hold_out_set = set(tuple(p) for p in hold_out)
    for p in all_possible:
        tp = tuple(p)
        if tp in adj_set: continue
        if tp in hold_out_set: test.append(p)
        else: non_adj.append(p)
    adj = np.asarray(adj, dtype=np.int64)
    non_adj = np.asarray(non_adj, dtype=np.int64)
    test = np.asarray(test, dtype=np.int64)
    def get_reward(p):
        return [1, 0] if p[0] > p[1] else [0, 1]
    adj_rew = np.asarray([get_reward(p) for p in adj], dtype=np.int64)
    non_adj_rew = np.asarray([get_reward(p) for p in non_adj], dtype=np.int64)
    test_rew = np.asarray([get_reward(p) for p in test], dtype=np.int64)
    return adj, non_adj, test, adj_rew, non_adj_rew, test_rew

def make_circular_pairs(n_stim, hold_out=None):
    all_possible = []
    for i in range(n_stim):
        for j in range(n_stim):
            if i != j: all_possible.append([i, j])
    def get_circular_reward(p):
        x, y = min(p), max(p)
        dx = y - x
        dy = (x - y) % n_stim
        if dx < dy:
            return [1, 0] if p[0] == y else [0, 1]
        else:
            return [1, 0] if p[0] == x else [0, 1]
    adj = []
    for i in range(n_stim):
        adj.append([i, (i + 1) % n_stim])
        adj.append([(i + 1) % n_stim, i])
    adj = np.asarray(adj, dtype=np.int64)
    adj_set = set(tuple(p) for p in adj)
    non_adj = []
    test = []
    if hold_out is None:
        hold_out = [(0, n_stim // 2), (n_stim // 2, 0)]
    hold_out_set = set(tuple(p) for p in hold_out)
    for p in all_possible:
        tp = tuple(p)
        if tp in adj_set: continue
        if tp in hold_out_set: test.append(p)
        else: non_adj.append(p)
    adj = np.asarray(adj, dtype=np.int64)
    non_adj = np.asarray(non_adj, dtype=np.int64)
    test = np.asarray(test, dtype=np.int64)
    adj_rew = np.asarray([get_circular_reward(p) for p in adj], dtype=np.int64)
    non_adj_rew = np.asarray([get_circular_reward(p) for p in non_adj], dtype=np.int64)
    test_rew = np.asarray([get_circular_reward(p) for p in test], dtype=np.int64)
    return adj, non_adj, test, adj_rew, non_adj_rew, test_rew

# parameters
sides = 2
terms = 7
pairs, nonadjpairs, testpairs, pairs_reward, nonadjpairs_reward, testpairs_reward = make_linear_pairs(terms)
allpairs = np.concatenate((pairs, nonadjpairs, testpairs))
allpairs_reward = np.concatenate((pairs_reward, nonadjpairs_reward, testpairs_reward))
plen = len(pairs)
alen = len(allpairs)
maxvalue = 10
startstop = 2
noise = 0.
annealing = 0.
timesteps = 10**8
runs = 1000
rein1 = 4
rein2 = 4
punish1 = -11
punish2 = -11
inertia = 1
nsteps = 100
blocklength = nsteps*1


In [ ]:
start = time.perf_counter()
thrds = 3
sq1 = SeedSequence()
randomentropy = sq1.entropy
sg = SeedSequence(randomentropy)
rgs = numba.typed.List([Generator(PCG64DXSM(s)) for s in sg.spawn(runs)])
with concurrent.futures.ProcessPoolExecutor(max_workers=thrds) as executor:
    future_to_run = {executor.submit(play_sequence, r, rgs[r], rein1, punish1, rein2, punish2, timesteps, nsteps, sides, pairs, testpairs, nonadjpairs, allpairs, pairs_reward, testpairs_reward, nonadjpairs_reward, allpairs_reward, plen, alen, terms, maxvalue, startstop, noise, annealing, runs, inertia, blocklength): r for r in range(runs)}
    results = [None] * runs
    for future in concurrent.futures.as_completed(future_to_run):
        r = future_to_run[future]
        try:
            results[r] = future.result()
            if r % 10 == 0: print(f'finished run #{r}')
        except Exception as exc:
            print(f'generated an exception in run {r}: {exc}')
final_sigweights = np.asarray([res[0] for res in results])
final_cumsuc = np.asarray([res[1] for res in results])
final_cumsucadd = np.asarray([res[2] for res in results])
final_testcumsucadd = np.asarray([res[3] for res in results])
final_recweights = np.asarray([res[4] for res in results])
final_pair_stats = np.asarray([res[5] for res in results])
print("average cumsuc = ")
print(np.average(final_cumsuc)/runs)
print(" ")
print("average final cumsucadd = ")
print(np.average(final_cumsucadd)/(100))
print(" ")
print("average test cumsum = ")
print(np.average(final_testcumsucadd)/(100))
print(" ")
finish = time.perf_counter()
print(f'Finished in {round(finish-start,0)/60} minutes')

In [ ]:
distances = np.abs(testpairs[:, 0] - testpairs[:, 1])
unique_dists = np.unique(distances)
print("Success rate by distance:")
for d in unique_dists:
    mask = (distances == d)
    attempts = np.sum(final_pair_stats[:, mask, 0])
    successes = np.sum(final_pair_stats[:, mask, 1])
    print(f"Distance {d}: {successes/attempts if attempts > 0 else 0:.4f}")
np.save('structure_noiseAnn_dsktp_2s_BASElearning-MV10_mod_sigweights', final_sigweights)
np.save('structure_noiseAnn_dsktp_2s_BASElearning-MV10_mod_recweights', final_recweights)
np.save('structure_noiseAnn_dsktp_2s_BASElearning-MV10_mod_testcumsucadd', final_testcumsucadd)
np.save('structure_noiseAnn_dsktp_2s_BASElearning-MV10_mod_pair_stats', final_pair_stats)

In [ ]:
cutoff = 90
final_test_count = 0
for cumsum in final_testcumsucadd:
    if cumsum > cutoff: final_test_count += 1
print(final_test_count)

In [ ]:
@numba.njit
def duplicates(dbins):
    dup = 0
    for iii in range(len(dbins)):
        for jjj in range(len(dbins)):
            if iii != jjj:
                if (dbins[iii][0] == dbins[jjj][0]) & (dbins[iii][1] == dbins[jjj][1]):
                    dup = 1
    return dup
@numba.njit
def runs_dup_bins(runs, fsigweights, t):
    dup_count = 0
    for i0 in range(runs):
        sw = fsigweights[i0].copy()
        swl = sw.argmax(axis=3)[0, 0]
        swr = sw.argmax(axis=3)[0, 1]
        bns = np.zeros(((2*(t-1)), 2), dtype = numba.int64)
        for idx000 in range(t-1):
            bns[idx000][0] = swl[idx000] 
            bns[idx000][1] = swr[idx000+1]
        for idx001 in range(t-1):
            bns[t-1+idx001][0] = swl[idx001+1] 
            bns[t-1+idx001][1] = swr[idx001]
        dup_count += duplicates(bns)
    return dup_count

In [ ]:
runs_dup_bins(runs, final_sigweights, terms)